# Portfolio Risk Analysis
### Snowflake Notebook Demo

This notebook demonstrates mixed SQL + Python analytics on a subprime auto loan portfolio — all running live against governed Snowflake data.

**Key points:**
- No data imports — queries run directly in Snowflake
- Python + SQL in the same notebook — no separate compute
- Governance (masking, tags) applies automatically
- Cortex AI functions available natively
- All configuration in `config.py` — edit once for your own data

In [12]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import config as cfg

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    _env = "Snowflake Notebook"
except Exception:
    from snowflake.snowpark import Session
    conn_name = os.getenv("SNOWFLAKE_CONNECTION_NAME", cfg.CONNECTION_NAME)
    session = Session.builder.configs({"connection_name": conn_name}).create()
    session.sql(f"USE DATABASE {cfg.DATABASE}").collect()
    session.sql(f"USE SCHEMA {cfg.SCHEMA}").collect()
    _env = f"Local (connection: {conn_name})"

def sf_query(sql):
    return session.sql(sql).to_pandas()

print(f"Connected to Snowflake via {_env} | Database: {cfg.DATABASE} | Schema: {cfg.SCHEMA}")

Connected to Snowflake via Snowflake Notebook | Database: PPTX_TEST | Schema: GOLD


## 1. Portfolio Risk Summary
Pull the single-row portfolio summary from the Gold layer — this is a Dynamic Table that refreshes automatically.

In [13]:
summary = sf_query(f"SELECT * FROM {cfg.SUMMARY_TABLE}")
summary.T

,0
TOTAL_LOANS,5.000000e+03
TOTAL_FUNDED_AMOUNT,1.250000e+08
TOTAL_OUTSTANDING_BALANCE,9.800000e+07
TOTAL_COLLECTED,2.700000e+07
WEIGHTED_AVG_CREDIT_SCORE,6.425000e+02
WEIGHTED_AVG_APR,1.875000e+01
CURRENT_PCT,7.850000e+01
DPD_30_PCT,8.200000e+00
DPD_60_PCT,4.100000e+00
PROJECTED_LOSSES,6.125000e+06


## 2. Delinquency Distribution
Visualize how accounts are distributed across delinquency buckets.

In [14]:
collections = sf_query(f"""
    SELECT * FROM {cfg.COLLECTIONS_TABLE}
    ORDER BY CASE DELINQUENCY_BUCKET
        WHEN 'CURRENT' THEN 1 WHEN '30_DPD' THEN 2
        WHEN '60_DPD' THEN 3 WHEN '90_DPD' THEN 4
        WHEN '120_PLUS_DPD' THEN 5 WHEN 'CHARGEOFF' THEN 6
    END
""")

colors = {"CURRENT": "#29B5E8", "30_DPD": "#FFBE2E", "60_DPD": "#FF6F61",
          "90_DPD": "#D94032", "120_PLUS_DPD": "#8B1A1A", "CHARGEOFF": "#4A0404"}

fig_collections = px.bar(
    collections, x="DELINQUENCY_BUCKET", y="TOTAL_ACCOUNTS",
    color="DELINQUENCY_BUCKET", color_discrete_map=colors,
    text="TOTAL_ACCOUNTS",
    title="Account Distribution by Delinquency Bucket",
)
fig_collections.update_layout(showlegend=False, xaxis_title="", yaxis_title="Accounts", height=400)
fig_collections.update_traces(textposition="outside")
fig_collections.show()

## 3. Vintage Curve Analysis
Analyze cumulative net loss rates by origination month and risk tier — the core metric for ABS investors.

In [15]:
perf = sf_query(f"""
    SELECT ORIGINATION_MONTH, RISK_TIER,
        SUM(LOAN_COUNT) AS LOANS,
        SUM(TOTAL_FUNDED) AS FUNDED,
        SUM(DELINQUENT_LOANS) AS DELINQUENT,
        ROUND(AVG(AVG_CREDIT_SCORE), 0) AS AVG_SCORE
    FROM {cfg.PERFORMANCE_TABLE}
    GROUP BY ORIGINATION_MONTH, RISK_TIER
    ORDER BY ORIGINATION_MONTH, RISK_TIER
""")

perf["DELINQUENCY_RATE"] = (perf["DELINQUENT"] / perf["LOANS"] * 100).round(2)

tier_colors = {"SUBPRIME": "#D94032", "NEAR_PRIME": "#FF6F61", "PRIME": "#FFBE2E", "SUPER_PRIME": "#29B5E8"}

fig_vintage = px.line(
    perf, x="ORIGINATION_MONTH", y="DELINQUENCY_RATE",
    color="RISK_TIER", color_discrete_map=tier_colors,
    markers=True,
    title="Delinquency Rate by Vintage Month and Risk Tier",
)
fig_vintage.update_layout(
    xaxis_title="Origination Month", yaxis_title="Delinquency Rate (%)",
    legend_title="Risk Tier", height=450,
)
fig_vintage.show()

## 4. Risk Tier Heatmap
Credit score vs funded amount by risk tier — shows portfolio concentration.

In [16]:
tier_summary = perf.groupby("RISK_TIER", as_index=False).agg(
    TOTAL_LOANS=("LOANS", "sum"),
    TOTAL_FUNDED=("FUNDED", "sum"),
    AVG_SCORE=("AVG_SCORE", "mean"),
    AVG_DELINQUENCY=("DELINQUENCY_RATE", "mean"),
)

fig_scatter = px.scatter(
    tier_summary, x="AVG_SCORE", y="AVG_DELINQUENCY",
    size="TOTAL_FUNDED", color="RISK_TIER",
    color_discrete_map=tier_colors,
    text="RISK_TIER",
    title="Risk Tier: Avg Credit Score vs Delinquency Rate (bubble = funded amount)",
)
fig_scatter.update_traces(textposition="top center")
fig_scatter.update_layout(
    xaxis_title="Avg Credit Score", yaxis_title="Avg Delinquency Rate (%)",
    showlegend=False, height=450,
)
fig_scatter.show()

## 5. Cortex AI: Natural Language Portfolio Summary
Use Snowflake Cortex COMPLETE to generate an executive summary from the data — no external API keys, no data leaves Snowflake.

In [17]:
ai_result = sf_query(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        '{cfg.CORTEX_MODEL}',
        'You are a risk analyst at a subprime auto lender. Write a concise executive summary (5 bullet points) of this portfolio. Include specific numbers. Data: ' ||
        (SELECT OBJECT_CONSTRUCT(
            'total_loans', TOTAL_LOANS,
            'total_funded', TOTAL_FUNDED_AMOUNT,
            'outstanding', TOTAL_OUTSTANDING_BALANCE,
            'collected', TOTAL_COLLECTED,
            'avg_credit_score', WEIGHTED_AVG_CREDIT_SCORE,
            'avg_apr', WEIGHTED_AVG_APR,
            'current_pct', CURRENT_PCT,
            'dpd_30_pct', DPD_30_PCT,
            'dpd_60_pct', DPD_60_PCT,
            'projected_losses', PROJECTED_LOSSES,
            'loss_reserve_pct', LOSS_RESERVE_PCT
        )::VARCHAR FROM {cfg.SUMMARY_TABLE})
    ) AS AI_SUMMARY
""")

ai_text = ai_result.iloc[0]["AI_SUMMARY"]

from IPython.display import Markdown, display
display(Markdown("### AI-Generated Portfolio Summary\n\n" + ai_text))

### AI-Generated Portfolio Summary

# Executive Summary: Subprime Auto Loan Portfolio

- **Portfolio Scale & Yield:** The portfolio comprises **5,000 loans** with **$125.0M total funded**, generating an average APR of **18.75%** against a borrower base with a mean credit score of **642.5** — consistent with deep subprime/near-prime risk profile expectations.

- **Collection Performance:** Of the **$98.0M outstanding balance**, **$27.0M (21.6%)** has been collected to date, with **78.5% of loans current** — a current rate that, while acceptable for the segment, leaves **21.5% of the book in some stage of stress or delinquency.**

- **Delinquency Deterioration:** Early-stage delinquency (30+ DPD) stands at **8.2%** with late-stage (60+ DPD) at **4.1%**, representing a **50% roll-through rate** from 30 to 60 days — a concerning migration pattern suggesting limited borrower recovery once delinquency begins.

- **Loss Exposure:** Projected losses are estimated at **$6.125M**, supported by a loss reserve of **6.25% ($6.125M)** against outstanding balances. While reserves appear fully funded *today*, accelerating 60+ DPD migration could rapidly erode this coverage with **zero buffer margin.**

- **Risk Outlook:** The combination of a sub-650 average credit score, a 4.1% severe delinquency rate, and a razor-thin **1.0x reserve coverage ratio** warrants immediate review of provisioning adequacy and tightened underwriting criteria on new originations to protect net portfolio yield.

## 6. Simple Risk Scoring Model
Quick logistic regression on credit score to predict delinquency — shows how data science happens inside the platform.

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

loans = sf_query(f"""
    SELECT AVG_CREDIT_SCORE, AVG_APR, LOAN_COUNT, DELINQUENT_LOANS,
        CASE WHEN DELINQUENCY_RATE_PCT > 5 THEN 1 ELSE 0 END AS IS_DELINQUENT
    FROM {cfg.PERFORMANCE_TABLE}
""")

X = loans[["AVG_CREDIT_SCORE", "AVG_APR"]].astype(float).values
y = loans["IS_DELINQUENT"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Risk Scoring Model Performance:\n")
print(classification_report(y_test, y_pred, target_names=["Not Delinquent", "Delinquent"]))

scores = np.linspace(550, 850, 100)
probs = model.predict_proba(np.column_stack([scores, np.full(100, 10.0)]))[:, 1]

fig_model = go.Figure()
fig_model.add_trace(go.Scatter(x=scores, y=probs * 100, mode="lines", line=dict(color="#D94032", width=3)))
fig_model.update_layout(
    title="Predicted Delinquency Probability by Credit Score",
    xaxis_title="Credit Score", yaxis_title="Delinquency Probability (%)",
    height=400,
)
fig_model.show()

Risk Scoring Model Performance:

                precision    recall  f1-score   support

Not Delinquent       1.00      1.00      1.00         7
    Delinquent       1.00      1.00      1.00        11

      accuracy                           1.00        18
     macro avg       1.00      1.00      1.00        18
  weighted avg       1.00      1.00      1.00        18



---
**Key Takeaway:** Everything in this notebook — SQL queries, Python analytics, AI-generated summaries, and ML models — runs directly against governed Snowflake data. No data copies, no external compute, no ungoverned access.

## 7. Export to PowerPoint

Run this cell after completing the analysis above. It builds a 7-slide deck with native PowerPoint charts using `python-pptx`.

**Snowflake Notebooks:** Replace `prs.save(output_path)` with a stage `PUT` and `GET_PRESIGNED_URL` for download.

In [ ]:
import io
import datetime
import numpy as np
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION
from pptx.chart.data import CategoryChartData, BubbleChartData
from lxml import etree

PBLUE = RGBColor(0x29, 0xB5, 0xE8)
PDARK = RGBColor(0x26, 0x27, 0x30)
PWHITE = RGBColor(0xFF, 0xFF, 0xFF)
PLGRAY = RGBColor(0xF0, 0xF2, 0xF6)
PRED = RGBColor(0xD9, 0x40, 0x32)
PGREEN = RGBColor(0x2E, 0xCC, 0x71)
PAMBER = RGBColor(0xFF, 0xBE, 0x2E)
PMGRAY = RGBColor(0x88, 0x88, 0x88)

TIER_PPTX_COLORS = {
    "PRIME": PBLUE, "NEAR_PRIME": PAMBER, "SUBPRIME": PRED,
    "SUPER_PRIME": PGREEN, "DEEP_SUBPRIME": RGBColor(0xFF, 0x6B, 0x6B),
}

nsmap = {"c": "http://schemas.openxmlformats.org/drawingml/2006/chart",
         "a": "http://schemas.openxmlformats.org/drawingml/2006/main"}


def _add_rect(slide, l, t, w, h, rgb):
    s = slide.shapes.add_shape(1, Inches(l), Inches(t), Inches(w), Inches(h))
    s.line.fill.background()
    s.fill.solid()
    s.fill.fore_color.rgb = rgb
    return s


def _add_text(slide, text, l, t, w, h, size, bold=False, color=None, align=PP_ALIGN.LEFT, wrap=True):
    txb = slide.shapes.add_textbox(Inches(l), Inches(t), Inches(w), Inches(h))
    txb.word_wrap = wrap
    tf = txb.text_frame
    tf.word_wrap = wrap
    p = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.size = Pt(size)
    run.font.bold = bold
    if color:
        run.font.color.rgb = color
    return txb


def _slide_header(prs, header, sw):
    blank = prs.slide_layouts[6]
    s = prs.slides.add_slide(blank)
    _add_rect(s, 0, 0, sw, 0.55, PDARK)
    _add_text(s, header, 0.2, 0.05, sw - 0.4, 0.45, 15, bold=True, color=PWHITE)
    return s


def _set_point_color(series_element, idx, rgb):
    dPt = etree.SubElement(series_element, etree.QName(nsmap["c"], "dPt"))
    idx_el = etree.SubElement(dPt, etree.QName(nsmap["c"], "idx"))
    idx_el.set("val", str(idx))
    spPr = etree.SubElement(dPt, etree.QName(nsmap["c"], "spPr"))
    solidFill = etree.SubElement(spPr, etree.QName(nsmap["a"], "solidFill"))
    srgbClr = etree.SubElement(solidFill, etree.QName(nsmap["a"], "srgbClr"))
    srgbClr.set("val", str(rgb))


def _set_series_color(series, rgb):
    spPr = series._element.find(etree.QName(nsmap["c"], "spPr"))
    if spPr is None:
        spPr = etree.SubElement(series._element, etree.QName(nsmap["c"], "spPr"))
    for old in spPr.findall(etree.QName(nsmap["a"], "solidFill")):
        spPr.remove(old)
    solidFill = etree.SubElement(spPr, etree.QName(nsmap["a"], "solidFill"))
    srgbClr = etree.SubElement(solidFill, etree.QName(nsmap["a"], "srgbClr"))
    srgbClr.set("val", str(rgb))
    ln = spPr.find(etree.QName(nsmap["a"], "ln"))
    if ln is None:
        ln = etree.SubElement(spPr, etree.QName(nsmap["a"], "ln"))
    for old in ln.findall(etree.QName(nsmap["a"], "solidFill")):
        ln.remove(old)
    ln_fill = etree.SubElement(ln, etree.QName(nsmap["a"], "solidFill"))
    ln_clr = etree.SubElement(ln_fill, etree.QName(nsmap["a"], "srgbClr"))
    ln_clr.set("val", str(rgb))


print("Building editable PowerPoint with native charts...")

row = summary.iloc[0]
total_funded_m = float(row["TOTAL_FUNDED_AMOUNT"]) / 1_000_000
avg_delinquency = round(float(perf["DELINQUENCY_RATE"].mean()), 1)

SW = 13.333
prs = Presentation()
prs.slide_width = Inches(SW)
prs.slide_height = Inches(7.5)

s = prs.slides.add_slide(prs.slide_layouts[6])
_add_rect(s, 0, 0, SW, 7.5, PBLUE)
_add_rect(s, 0, 5.8, SW, 1.7, PDARK)
_add_text(s, cfg.APP_TITLE, 0.7, 1.6, SW - 1.4, 1.1, 40, bold=True, color=PWHITE)
_add_text(s, "Exploratory Data Analysis \u2014 Snowflake Notebook", 0.7, 2.75, SW - 1.4, 0.7, 22, color=PWHITE)
_add_text(s, datetime.date.today().strftime("%B %Y"), 0.7, 3.55, 4, 0.5, 16, color=RGBColor(0xCC, 0xEE, 0xF8))
_add_text(s, "Powered by Snowflake Dynamic Tables + Cortex AI", 0.7, 6.15, SW - 1.4, 0.5, 13, color=PMGRAY)

s = prs.slides.add_slide(prs.slide_layouts[6])
_add_rect(s, 0, 0, SW, 0.55, PDARK)
_add_text(s, "Portfolio KPIs", 0.2, 0.05, SW - 0.4, 0.45, 15, bold=True, color=PWHITE)
kpis_r1 = [
    ("Total loans", f"{int(row['TOTAL_LOANS']):,}", PBLUE),
    ("Total funded", f"${total_funded_m:.2f}M", PBLUE),
    ("Outstanding", f"${float(row['TOTAL_OUTSTANDING_BALANCE'])/1e6:.2f}M", PBLUE),
    ("Avg credit score", f"{int(row['WEIGHTED_AVG_CREDIT_SCORE'])}", PBLUE),
]
kpis_r2 = [
    ("Current %", f"{row['CURRENT_PCT']}%", PGREEN),
    ("30+ DPD", f"{row['DPD_30_PCT']}%", PAMBER),
    ("Avg delinquency", f"{avg_delinquency}%", PRED),
    ("Projected losses", f"${float(row['PROJECTED_LOSSES']):,.0f}", PRED),
]
box_w = (SW - 0.36 - 3 * 0.13) / 4
box_gap = 0.13
for row_kpis, box_top in [(kpis_r1, 0.72), (kpis_r2, 1.85)]:
    for i, (label, value, val_color) in enumerate(row_kpis):
        lx = 0.18 + i * (box_w + box_gap)
        _add_rect(s, lx, box_top, box_w, 0.95, PLGRAY)
        _add_text(s, value, lx+0.1, box_top+0.04, box_w-0.2, 0.5, 20, bold=True, color=val_color, align=PP_ALIGN.CENTER)
        _add_text(s, label, lx+0.1, box_top+0.56, box_w-0.2, 0.3, 9, color=PDARK, align=PP_ALIGN.CENTER)

bucket_order = ["CURRENT", "30_DPD", "60_DPD", "90_DPD", "120+_DPD"]
cdf = collections.copy()
cdf["sort_key"] = cdf["DELINQUENCY_BUCKET"].map({b: i for i, b in enumerate(bucket_order)})
cdf = cdf.sort_values("sort_key")
bucket_colors = []
for b in cdf["DELINQUENCY_BUCKET"]:
    if b == "CURRENT":
        bucket_colors.append(PGREEN)
    elif "30" in b:
        bucket_colors.append(PAMBER)
    else:
        bucket_colors.append(PRED)

chart_data = CategoryChartData()
chart_data.categories = list(cdf["DELINQUENCY_BUCKET"])
chart_data.add_series("Total Accounts", list(cdf["TOTAL_ACCOUNTS"].astype(float)))

s3 = _slide_header(prs, "Delinquency Distribution by Bucket", SW)
chart_frame = s3.shapes.add_chart(
    XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(0.18), Inches(0.65), Inches(SW - 0.36), Inches(6.6), chart_data)
chart = chart_frame.chart
chart.has_legend = False
plot = chart.plots[0]
plot.gap_width = 80
series_el = plot.series[0]._element
for idx, clr in enumerate(bucket_colors):
    _set_point_color(series_el, idx, clr)
plot.series[0].data_labels.show_value = True
plot.series[0].data_labels.font.size = Pt(9)
plot.series[0].data_labels.font.bold = True
plot.series[0].data_labels.number_format = "#,##0"

tiers = sorted(perf["RISK_TIER"].dropna().unique())
months = sorted(perf["ORIGINATION_MONTH"].dropna().unique())
month_labels = [str(m)[:10] if hasattr(m, 'strftime') else str(m) for m in months]

line_data = CategoryChartData()
line_data.categories = month_labels
for tier in tiers:
    td = perf[perf["RISK_TIER"] == tier].copy()
    td_grouped = td.groupby("ORIGINATION_MONTH")["DELINQUENCY_RATE"].mean()
    vals = []
    for m in months:
        if m in td_grouped.index:
            vals.append(float(td_grouped[m]))
        else:
            vals.append(None)
    line_data.add_series(tier, vals)

s4 = _slide_header(prs, "Delinquency Rate by Vintage Month and Risk Tier", SW)
chart_frame = s4.shapes.add_chart(
    XL_CHART_TYPE.LINE_MARKERS, Inches(0.18), Inches(0.65), Inches(SW - 0.36), Inches(6.6), line_data)
chart = chart_frame.chart
chart.has_legend = True
chart.legend.position = XL_LEGEND_POSITION.BOTTOM
chart.legend.include_in_layout = False
chart.legend.font.size = Pt(8)
for i, tier in enumerate(tiers):
    ser = chart.series[i]
    clr = TIER_PPTX_COLORS.get(tier, PMGRAY)
    _set_series_color(ser, clr)
    ser.smooth = False
    ser.format.line.width = Pt(2)

scatter_data = sf_query(f"""
    SELECT RISK_TIER,
           ROUND(AVG(AVG_CREDIT_SCORE), 0) AS AVG_CREDIT_SCORE,
           ROUND(AVG(DELINQUENCY_RATE_PCT), 2) AS AVG_DELINQUENCY_RATE,
           SUM(TOTAL_FUNDED) AS TOTAL_FUNDED
    FROM {cfg.PERFORMANCE_TABLE}
    GROUP BY RISK_TIER
""")

bubble_data = BubbleChartData()
for _, brow in scatter_data.iterrows():
    tier_name = brow["RISK_TIER"]
    ser = bubble_data.add_series(tier_name)
    ser.add_data_point(float(brow["AVG_CREDIT_SCORE"]), float(brow["AVG_DELINQUENCY_RATE"]), float(brow["TOTAL_FUNDED"]))

s5 = _slide_header(prs, "Risk Tier: Credit Score vs Delinquency Rate", SW)
chart_frame = s5.shapes.add_chart(
    XL_CHART_TYPE.BUBBLE, Inches(0.18), Inches(0.65), Inches(SW - 0.36), Inches(6.6), bubble_data)
chart = chart_frame.chart
chart.has_legend = True
chart.legend.position = XL_LEGEND_POSITION.BOTTOM
chart.legend.include_in_layout = False
chart.legend.font.size = Pt(8)
for i, (_, brow) in enumerate(scatter_data.iterrows()):
    clr = TIER_PPTX_COLORS.get(brow["RISK_TIER"], PMGRAY)
    _set_series_color(chart.series[i], clr)

scores_arr = np.linspace(550, 850, 30)
probs_arr = model.predict_proba(np.column_stack([scores_arr, np.full(30, 10.0)]))[:, 1] * 100
prob_data = CategoryChartData()
prob_data.categories = [str(int(s)) for s in scores_arr]
prob_data.add_series("Delinquency Probability %", [round(float(p), 2) for p in probs_arr])

s6 = _slide_header(prs, "Predicted Delinquency Probability by Credit Score", SW)
chart_frame = s6.shapes.add_chart(
    XL_CHART_TYPE.LINE, Inches(0.18), Inches(0.65), Inches(SW - 0.36), Inches(6.6), prob_data)
chart = chart_frame.chart
chart.has_legend = False
_set_series_color(chart.series[0], PRED)
chart.series[0].smooth = True
chart.series[0].format.line.width = Pt(3)

try:
    _ai_slide_text = ai_text
except NameError:
    _ai_slide_text = "Run the Cortex AI cell before exporting to include the AI-generated summary."

s = prs.slides.add_slide(prs.slide_layouts[6])
_add_rect(s, 0, 0, SW, 0.55, PDARK)
_add_text(s, "AI-Generated Executive Summary (Snowflake Cortex)", 0.2, 0.05, SW - 0.4, 0.45, 15, bold=True, color=PWHITE)
_add_text(s, _ai_slide_text, 0.3, 0.75, SW - 0.6, 6.5, 11, color=PDARK, wrap=True)

buf = io.BytesIO()
prs.save(buf)
buf.seek(0)
pptx_bytes = buf.getvalue()

slide_manifest = [
    ("Title Slide", "layout"),
    ("Portfolio KPIs", "kpi_boxes"),
    ("Delinquency Distribution by Bucket", "COLUMN_CLUSTERED"),
    ("Delinquency Rate by Vintage Month and Risk Tier", "LINE_MARKERS"),
    ("Risk Tier: Credit Score vs Delinquency Rate", "BUBBLE"),
    ("Predicted Delinquency Probability by Credit Score", "LINE (smooth)"),
    ("AI-Generated Executive Summary (Snowflake Cortex)", "text"),
]
print(f"Built editable presentation: {len(pptx_bytes)/1024:.0f} KB | {len(prs.slides)} slides | {SW}\" x 7.5\" (16:9)")
print(f"Charts are native OOXML objects \u2014 fully editable in PowerPoint / Google Slides\n")
print(f"{'#':<4} {'Slide Title':<55} {'Type':<20}")
print("-" * 79)
for i, (title, stype) in enumerate(slide_manifest):
    print(f"{i+1:<4} {title:<55} {stype:<20}")

Building editable PowerPoint with native charts...


KeyError: 'DELINQUENCY_RATE_PCT'

## Persist PowerPoint to Snowflake

The analysis above generates a PowerPoint deck (`portfolio_analysis.pptx`) containing:

1. **Title slide** — Portfolio Risk Analysis
2. **Portfolio KPIs** — Total loans, funded amounts, delinquency buckets
3. **Delinquency Distribution** — Collections breakdown by bucket
4. **Vintage Curves** — Delinquency rate by origination month and risk tier
5. **Risk Tier Scatter** — Credit score vs delinquency rate by tier
6. **Risk Scoring Model** — Logistic regression predicting delinquency probability
7. **AI Executive Summary** — Generated via Snowflake Cortex AI

The cell below uploads the file to your configured stage so it is persisted in Snowflake and accessible via `DIRECTORY()` or `GET`.

In [ ]:
tmp_path = f'/tmp/{cfg.DOWNLOAD_PREFIX}.pptx'
with open(tmp_path, 'wb') as f:
    f.write(pptx_bytes)

session.file.put(tmp_path, cfg.STAGE, auto_compress=False, overwrite=True)
session.sql(f"ALTER STAGE {cfg.STAGE.lstrip('@')} REFRESH").collect()

filename = f'{cfg.DOWNLOAD_PREFIX}.pptx'
url_df = session.sql(f"SELECT GET_PRESIGNED_URL({cfg.STAGE}, '{filename}', 3600) AS URL").to_pandas()
presigned_url = url_df["URL"].iloc[0]

result = session.sql(f"SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED FROM DIRECTORY({cfg.STAGE}) WHERE RELATIVE_PATH LIKE '%pptx%'").to_pandas()
print("Persisted to Snowflake stage:")
print(result.to_string(index=False))
print(f"\nPresigned download URL (expires in 1 hour):\n{presigned_url}")

from IPython.display import display, HTML, FileLink
display(HTML(f'<a href="{presigned_url}" target="_blank" style="font-size:14px; padding:8px 16px; background:#29B5E8; color:white; border-radius:6px; text-decoration:none;">Download from Snowflake Stage</a>'))
try:
    display(FileLink(tmp_path))
except Exception:
    pass

Persisted to Snowflake stage:
          RELATIVE_PATH  SIZE             LAST_MODIFIED
portfolio_analysis.pptx 64539 2026-04-21 08:51:58-07:00


## Stage Directory Listing

List all files currently stored in the configured Snowflake stage.

In [ ]:
session.sql(f"ALTER STAGE {cfg.STAGE.lstrip('@')} REFRESH").collect()
stage_files = session.sql(f"""
    SELECT RELATIVE_PATH,
           SIZE,
           LAST_MODIFIED,
           FILE_URL,
           ETAG
    FROM DIRECTORY({cfg.STAGE})
    ORDER BY LAST_MODIFIED DESC
""").to_pandas()

print(f"Stage: {cfg.STAGE}")
print(f"Total files: {len(stage_files)}")
print(f"Total size: {stage_files['SIZE'].sum() / 1024:.1f} KB\n")
print(stage_files.to_string(index=False))